### Validate that the new dataset is compatible with the old dataset
* Ensure that both datasets have the same columns
* Make several plots of all of the variables

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import uproot
from datetime import datetime
import plotly.graph_objects as go
import re
import awkward as ak
import pandas as pd

pd.set_option('display.max_rows', None)

In [ ]:
# new: https://github.com/cms-lpc-llp/run3_llp_analyzer/blob/main/lists/MDSNano/v2/MC_Sumto4B_MH-125-MS-15-ctauS-1000_TuneCP5_13p6TeV_powheg-pythia8.txt
# old: https://gitlab.nrp-nautilus.io/aaportel/mds-ml/-/blob/main/data/data-paths/ggH_dirs.txt?ref_type=heads#L100-L158


NEW_DATA_FILE = '/uscms/home/tlee/nobackup/work/mds-ml/data/MuonSystem_Tree.root' 
NEW_RAW_DATA_FILE = '/uscms/home/tlee/nobackup/work/mds-ml/data/samples/hidden_valley/PAT_NANO_100.root' 
OLD_DATA_FILE = '/uscms/home/tlee/nobackup/work/mds-ml/data/samples/ggH/displacedJetMuon_ntupler_1.root' 

OUTPUT_DIR = '/uscms/home/tlee/nobackup/work/mds-ml/notebooks/model_compatibility'
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

old_root_file = uproot.open(OLD_DATA_FILE)
old_tree = old_root_file['ntuples']['llp']

new_root_file = uproot.open(NEW_DATA_FILE)
new_tree = new_root_file['MuonSystem']

new_raw_root_file = uproot.open(NEW_RAW_DATA_FILE)
new_raw_tree = new_raw_root_file['Events']

old_tree_keys = old_tree.keys()
new_tree_keys = new_tree.keys()
new_raw_tree_keys = new_raw_tree.keys()

# Pad the shorter lists to the same length
max_len = max(len(old_tree_keys), len(new_tree_keys), len(new_raw_tree_keys))
old_tree_keys += [''] * (max_len - len(old_tree_keys))
new_tree_keys += [''] * (max_len - len(new_tree_keys))
new_raw_tree_keys += [''] * (max_len - len(new_raw_tree_keys))


df = pd.DataFrame({
    'Old Tree (llp)': old_tree_keys,
    'New Raw Tree (Events)': new_raw_tree_keys,
    'New Tree (MuonSystem)': new_tree_keys,
})

df.head(df.shape[0])


,Old Tree (llp),New Raw Tree (Events),New Tree (MuonSystem)
0,isData,run,runNum
1,nPV,luminosityBlock,MC_condition
2,runNum,event,lumiSec
3,lumiNum,bunchCrossing,evtNum
4,eventNum,HTXS_njets25,mH
5,eventTime,HTXS_njets30,mX
6,pvX,HTXS_stage1_1_cat_pTjet25GeV,ctau
7,pvY,HTXS_stage1_1_cat_pTjet30GeV,HLT_CSCCSC
8,pvZ,HTXS_stage1_1_fine_cat_pTjet25GeV,HLT_CSCDT
9,fixedGridRhoAll,HTXS_stage1_1_fine_cat_pTjet30GeV,HLT_CscCluster100_PNetTauhPFJet10_Loose


In [3]:
# Helper function to normalize keys: lowercase and strip special characters
def normalize_keys(keys):
    return {re.sub(r'\W+', '', key.lower()): key for key in keys}

# Normalize the keys
old_keys_normalized = normalize_keys(old_tree_keys)
new_keys_normalized = normalize_keys(new_tree_keys)

# Find intersection and differences
common_keys = set(old_keys_normalized.keys()) & set(new_keys_normalized.keys())
only_in_old = set(old_keys_normalized.keys()) - set(new_keys_normalized.keys())
only_in_new = set(new_keys_normalized.keys()) - set(old_keys_normalized.keys())

# Prepare report content
report_lines = []

report_lines.append(f"Common keys ({len(common_keys)}):")
for key in sorted(common_keys):
    report_lines.append(f" - Old: {old_keys_normalized[key]} | New: {new_keys_normalized[key]}")

report_lines.append(f"\nKeys only in OLD tree ({len(only_in_old)}):")
for key in sorted(only_in_old):
    report_lines.append(f" - {old_keys_normalized[key]}")

report_lines.append(f"\nKeys only in NEW tree ({len(only_in_new)}):")
for key in sorted(only_in_new):
    report_lines.append(f" - {new_keys_normalized[key]}")

# Join all lines
report_text = '\n'.join(report_lines)


In [4]:
with open(f"{OUTPUT_DIR}/model_compatibility.txt", 'w') as f:
    f.write(report_text)

print(f"\nReport saved to: {OUTPUT_DIR}/model_compatibility.txt")


Report saved to: /uscms/home/tlee/nobackup/work/mds-ml/notebooks/model_compatibility/model_compatibility.txt
